# B3b Defense – 07 LLM Macroeconomic Analysis

**Objective:** Send the XGBoost forecast results (`defense_forecast_2026_sac.csv`) and
the SHAP driver breakdown (`defense_forecast_2026_drivers_sac.csv`) to Claude Sonnet 4.6
and request a macroeconomic interpretation tailored to a company planning to enter the
US defense segment with new products in 2026.

The LLM acts as a macroeconomic analyst — it does not re-run any model, but contextualises
the quantitative outputs for strategic management communication.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import anthropic
from IPython.display import display, Markdown

# Force UTF-8 output on Windows (avoids CP1252 encoding errors)
if sys.stdout.encoding and sys.stdout.encoding.lower() != 'utf-8':
    sys.stdout.reconfigure(encoding='utf-8')

# Load API key from project .env
load_dotenv(dotenv_path='../../.env')

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        'ANTHROPIC_API_KEY not found. Add it to the project .env file: '
        'ANTHROPIC_API_KEY=sk-ant-...'
    )

client         = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
DATA_RAW       = '../data/raw/'
DATA_PROCESSED = '../data/processed/'
OUTPUT_DIR     = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Anthropic client initialised')

In [ ]:
# ===========================================================================
# 1. Load and compute ADEFNO statistics
# ===========================================================================
adefno_raw = pd.read_csv(DATA_RAW + 'ADEFNO.csv', parse_dates=['observation_date'])
adefno_raw = adefno_raw.rename(columns={'observation_date': 'date'}).sort_values('date')

# Last historical value = Dec 2025 (simulated current date: 31.12.2025)
adefno_hist = adefno_raw[adefno_raw['date'] <= '2025-12-31'].copy()
last_row     = adefno_hist.iloc[-1]
adefno_last_month = last_row['date'].strftime('%b %Y')
adefno_last_value = last_row['ADEFNO']

# 12-month average 2025 (Jan–Dec 2025)
adefno_2025 = adefno_hist[adefno_hist['date'].dt.year == 2025]['ADEFNO']
adefno_12m_avg = adefno_2025.mean()

# 2020–2025 range
adefno_2020_2025 = adefno_hist[adefno_hist['date'].dt.year >= 2020]['ADEFNO']
idx_min = adefno_2020_2025.idxmin()
idx_max = adefno_2020_2025.idxmax()
adefno_min_val  = adefno_2020_2025.loc[idx_min]
adefno_max_val  = adefno_2020_2025.loc[idx_max]
adefno_min_date = adefno_hist.loc[idx_min, 'date'].strftime('%b %Y')
adefno_max_date = adefno_hist.loc[idx_max, 'date'].strftime('%b %Y')

# ===========================================================================
# 2. Load and compute FDEFX statistics
# ===========================================================================
fdefx_raw = pd.read_csv(DATA_RAW + 'FDEFX.csv', parse_dates=['observation_date'])
fdefx_raw = fdefx_raw.rename(columns={'observation_date': 'date'}).sort_values('date')

fdefx_hist     = fdefx_raw[fdefx_raw['date'] <= '2025-12-31'].copy()
fdefx_last_row = fdefx_hist.iloc[-1]
fdefx_last_val = fdefx_last_row['FDEFX']
fdefx_last_qtr = fdefx_last_row['date']

q_num = (fdefx_last_qtr.month - 1) // 3 + 1
fdefx_last_quarter = f'Q{q_num} {fdefx_last_qtr.year}'

prior_date       = fdefx_last_qtr - pd.DateOffset(years=1)
fdefx_prior_row  = fdefx_hist[fdefx_hist['date'] == prior_date]
if fdefx_prior_row.empty:
    fdefx_prior_row = fdefx_hist.iloc[-5:-4]
fdefx_prior_val = float(fdefx_prior_row['FDEFX'].iloc[0])
fdefx_yoy       = (fdefx_last_val - fdefx_prior_val) / fdefx_prior_val * 100

# ===========================================================================
# 3. Load and compute IPB52300S statistics
# ===========================================================================
ipb_raw = pd.read_csv(DATA_RAW + 'IPB52300S.csv', parse_dates=['observation_date'])
ipb_raw = ipb_raw.rename(columns={'observation_date': 'date'}).sort_values('date')

ipb_hist      = ipb_raw[ipb_raw['date'] <= '2025-12-31'].copy()
ipb_last_row  = ipb_hist.iloc[-1]
ipb_last_val  = ipb_last_row['IPB52300S']
ipb_last_month_label = ipb_last_row['date'].strftime('%b %Y')

ipb_jan_val    = ipb_hist[ipb_hist['date'] == '2025-01-01']['IPB52300S'].iloc[0]
ipb_12m_change = ipb_last_val - ipb_jan_val
ipb_direction  = 'upward' if ipb_12m_change > 0 else 'downward'

# ===========================================================================
# 4. Load forecast and compute table
# ===========================================================================
forecast_df = pd.read_csv(DATA_PROCESSED + 'defense_forecast_2026_sac.csv')
forecast_df['date_parsed'] = pd.to_datetime(
    forecast_df['Date'].astype(str), format='%Y%m'
)
forecast_df['Month']        = forecast_df['date_parsed'].dt.strftime('%b %Y')
forecast_df['ADEFNO_USD_mn'] = (forecast_df['Net_Value_USD'] / 1e6).round(1)
annual_total_bn = (forecast_df['Net_Value_USD'].sum() / 1e9).round(1)

forecast_table = forecast_df[['Month', 'ADEFNO_USD_mn']].to_string(index=False)

# ===========================================================================
# 5. Load SHAP drivers and compute tables
# ===========================================================================
drivers_df   = pd.read_csv(DATA_PROCESSED + 'defense_forecast_2026_drivers_sac.csv')
feature_shap = drivers_df[~drivers_df['Driver'].isin(['base_value', 'prediction'])].copy()
feature_shap['abs_shap'] = feature_shap['SHAP_Value_USD_mn'].abs()

base_value_mn = float(
    drivers_df[drivers_df['Driver'] == 'base_value']['SHAP_Value_USD_mn'].iloc[0]
)

importance_df = (
    feature_shap.groupby('Driver')['abs_shap']
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
    .rename(columns={'abs_shap': 'Mean_Abs_SHAP_USD_mn'})
)
importance_df['Mean_Abs_SHAP_USD_mn'] = importance_df['Mean_Abs_SHAP_USD_mn'].round(1)
importance_table = importance_df.to_string(index=False)

monthly_top3 = (
    feature_shap.sort_values('abs_shap', ascending=False)
    .groupby('Date')
    .head(3)
    .sort_values(['Date', 'abs_shap'], ascending=[True, False])
    .copy()
)
monthly_top3['Month'] = pd.to_datetime(
    monthly_top3['Date'].astype(str), format='%Y%m'
).dt.strftime('%b %Y')
monthly_top3['SHAP_sign'] = monthly_top3['SHAP_Value_USD_mn'].apply(
    lambda x: f'+{x:.1f}' if x >= 0 else f'{x:.1f}'
)
monthly_top3_table = (
    monthly_top3[['Month', 'Driver', 'SHAP_sign']]
    .rename(columns={'SHAP_sign': 'SHAP_USD_mn'})
    .to_string(index=False)
)

print(f'ADEFNO last value:     {adefno_last_month}: {adefno_last_value:,.0f} USD mn')
print(f'ADEFNO 12m avg 2025:   {adefno_12m_avg:,.0f} USD mn')
print(f'ADEFNO range (2020–25): {adefno_min_val:,.0f} ({adefno_min_date}) – {adefno_max_val:,.0f} ({adefno_max_date})')
print(f'FDEFX last quarter:    {fdefx_last_quarter}: {fdefx_last_val:,.1f} USD bn (YoY {fdefx_yoy:+.1f}%)')
print(f'IPB52300S last:        {ipb_last_month_label}: {ipb_last_val:.2f} (12m change: {ipb_12m_change:+.2f}, {ipb_direction})')
print(f'Annual forecast total: {annual_total_bn} USD bn')
print(f'Model baseline:        {base_value_mn:,.1f} USD mn / month')

## Prompt Design

The system prompt positions Claude as a senior macroeconomic analyst writing for a corporate
controller. All numerical values in the prompt are computed programmatically from raw FRED CSVs
— the LLM is explicitly forbidden from adding any numbers not present in the prompt.

**Anti-hallucination rules injected into the user prompt:**
- Use **only** the numerical values explicitly provided (no memory-based statistics)
- Every factual claim not derivable from the tables must cite an **official primary source**
  with a verifiable URL (FRED, BEA, US Census, Federal Reserve, CBO, DoD Comptroller)
- No secondary sources; no industry association claims without a direct primary URL
- If a claim cannot be supported → **omit it entirely**
- Scenario probabilities flagged as **illustrative** unless sourced

**Output structure (4 sections + exec summary):**
1. Market Volume & Historical Context (ADEFNO trajectory, H1/H2 pattern, annual total in range context)
2. Driver Interpretation (AR Momentum, FDEFX Policy, IPB52300S Supply-Side — one paragraph each)
3. Monitoring Indicators (Markdown table: 4 KPIs with source URL, cadence, what to watch)
4. Key Risks & Model Limitation (US budget dynamics, carry-forward limitation)
5. Executive Summary (exactly 6 bullets, max 2 lines each — for SAC Planning Story)

In [ ]:
# ---------------------------------------------------------------------------
# Model selection — uncomment the model you want to use
# ---------------------------------------------------------------------------
# claude-sonnet-4-5   $3.00 input / $15.00 output per 1M tokens  (active)
MODEL = 'claude-sonnet-4-5'

# claude-haiku-4-5    $1.00 input / $5.00 output  per 1M tokens  (cheapest)
# MODEL = 'claude-haiku-4-5'

# claude-sonnet-4-6   $3.00 input / $15.00 output per 1M tokens  (latest)
# MODEL = 'claude-sonnet-4-6'

# claude-opus-4-6     $5.00 input / $25.00 output per 1M tokens  (most capable)
# MODEL = 'claude-opus-4-6'
# ---------------------------------------------------------------------------

MAX_TOKENS = 3500

# ---------------------------------------------------------------------------
# Prompts
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """\
You are a senior macroeconomic analyst specialising in US defense procurement markets.
Write for a corporate controller or enterprise planner who needs to defend forecast
assumptions in a management review. Prioritize verifiable sourcing over narrative
flourish. Brevity is a feature.\
"""

USER_PROMPT = f"""\
## Sourcing Rules — Read Before Answering

Use **only** the numerical values explicitly provided in this prompt.
Do not introduce any statistics, percentages, ratios, or historical figures from memory.

For every factual claim that is not directly derivable from the data tables below,
cite an **official primary source** with a specific, verifiable URL from this list:
  - FRED:            https://fred.stlouisfed.org
  - BEA:             https://www.bea.gov
  - US Census:       https://www.census.gov
  - Federal Reserve: https://www.federalreserve.gov
  - CBO:             https://www.cbo.gov
  - DoD Comptroller: https://comptroller.defense.gov

No secondary sources. No industry association claims without a direct primary URL.

If you cannot support a claim with either (a) the data provided or (b) an official
primary source with a verifiable URL, **omit the claim entirely**.

Flag any scenario probabilities, thresholds, or ranges as **illustrative** unless
they are derived from a cited primary source.

---

## Company Context

An industrial B2B manufacturer (compressors and pneumatic tools, Germany-based) is
launching new products in the US defense segment in 2026. The company has zero prior
defense revenue. Revenue targets are set in SAP Analytics Cloud (SAC) using a
management-defined market share assumption applied to the ML-forecasted market volume
below. The market share decision is made in SAC — this analysis covers market volume
and macro conditions only.

---

## Verified Input Data

### 1. ADEFNO — US Manufacturers' New Orders: Defense Capital Goods
Source: FRED series ADEFNO (https://fred.stlouisfed.org/series/ADEFNO)
Unit: USD millions, seasonally adjusted

Last observed value:   {adefno_last_month}: {adefno_last_value:,.0f} USD mn
12-month average 2025: {adefno_12m_avg:,.0f} USD mn  (Jan–Dec 2025, from CSV)
2020–2025 range:       {adefno_min_val:,.0f} USD mn ({adefno_min_date}) to {adefno_max_val:,.0f} USD mn ({adefno_max_date})

### 2. FDEFX — Federal Government: National Defense Consumption Expenditures & Gross Investment
Source: FRED series FDEFX (https://fred.stlouisfed.org/series/FDEFX)
Unit: USD billions, seasonally adjusted annual rate

Last observed value:    {fdefx_last_quarter}: {fdefx_last_val:,.1f} USD bn
Prior-year same quarter: {fdefx_prior_val:,.1f} USD bn
YoY change:             {fdefx_yoy:+.1f}%

### 3. IPB52300S — Industrial Production: Defense and Space Equipment
Source: FRED series IPB52300S (https://fred.stlouisfed.org/series/IPB52300S)
Unit: Index (2017=100), seasonally adjusted

Last observed value:  {ipb_last_month_label}: {ipb_last_val:.2f}
12-month trend 2025:  Jan 2025 = {ipb_jan_val:.2f} → Dec 2025 = {ipb_last_val:.2f} (change: {ipb_12m_change:+.2f} index pts, {ipb_direction})

### 4. XGBoost Forecast — ADEFNO Market Volume 2026
Model: rolling one-step-ahead XGBoost, trained on FRED macro data 2000–2025.
Macro features (IPB52300S, FDEFX) are carried forward at last known values (Dec 2025 / Q4 2025).
ADEFNO autoregressive features are updated each step from the prior predicted value.

Monthly forecast (USD mn):
{forecast_table}

Annual total 2026: {annual_total_bn} USD bn
Model baseline (SHAP expected value): {base_value_mn:,.1f} USD mn / month

### 5. SHAP Driver Breakdown
Unit: USD millions contribution per month (positive = above baseline, negative = below).
Computed via TreeExplainer on the trained XGBoost model.

Top-10 drivers by mean absolute SHAP contribution (averaged over all 12 forecast months):
{importance_table}

Monthly top-3 contributing drivers:
{monthly_top3_table}

Feature glossary:
  adefno_lag_1/2/3          Autoregressive lags of ADEFNO (t-1, t-2, t-3)
  ADEFNO_diff               Month-over-month change in ADEFNO at forecast origin
  ADEFNO_diff_lag_1..6      Lagged first differences of ADEFNO
  adefno_rolling_3m_mean    Rolling 3-month average of ADEFNO (prior values only)
  adefno_rolling_6m_mean    Rolling 6-month average of ADEFNO (prior values only)
  FDEFX                     Realized federal defense expenditures (quarterly, forward-filled)
  IPB52300S                 Defense & space industrial production index
  year                      Calendar year (captures secular trend)
  month / quarter / is_q4   Seasonal and calendar pattern features

---

## Analysis Request

Provide a structured analysis covering exactly the four sections below, then an
Executive Summary. Do not add sections. Do not discuss market share scenarios
(handled in SAC). Do not add geopolitical speculation beyond what the data supports.

---

### Section 1 — Market Volume & Historical Context

Using only the ADEFNO values provided above:
- Describe the 2026 forecast trajectory (H1 trough, H2 recovery, year-end level).
- Place the annual total ({annual_total_bn} USD bn) in context of the 2020–2025 range.
- Note the January spike and explain it as an artefact of the Dec 2025 outlier
  feeding into the AR structure, not a genuine demand signal.
- Keep to 3–4 short paragraphs.

### Section 2 — Driver Interpretation

Using only the SHAP table and feature glossary above:
- **AR Momentum** (adefno_lag_1, ADEFNO_diff, rolling means): What does the dominance
  of these features tell planners about the nature of defense order patterns?
- **FDEFX (Policy)**: What economic dynamic does realized government expenditure capture
  that AR lags cannot? Reference only the FDEFX values provided.
- **IPB52300S (Supply-Side)**: What does the production index add to the model?
  Reference only the IPB52300S values provided.
- One paragraph per driver category. No invented numbers.

### Section 3 — Monitoring Indicators for 2026

Provide a table of exactly 4 KPIs the planner must track quarterly to validate or
revise the forecast assumption. For each KPI:
  - Name & description
  - Official primary source with exact URL
  - Update cadence (monthly / quarterly)
  - What to watch for (one sentence, no invented thresholds)

Use Markdown table format.

### Section 4 — Key Risks & Model Limitation

Cover exactly two risk areas:

(a) **US Federal Budget Dynamics**: Discuss continuing resolution risk and debt-ceiling
    dynamics as they affect ADEFNO. Do not invent probability estimates.
    If you reference budget figures, cite the CBO or DoD Comptroller with a URL.

(b) **Carry-Forward Limitation**: Explain in plain language what it means that
    IPB52300S and FDEFX are frozen at their {ipb_last_month_label} / {fdefx_last_quarter}
    values for the entire 12-month horizon. What class of macro shock would most
    rapidly invalidate the forecast?

---

### Executive Summary

Provide exactly 6 bullet points, maximum 2 lines each. Written for direct use in a
SAC Planning Story or management slide. Bullets must cover:
  1. Core forecast message (annual total, trajectory)
  2. Strongest driver and what it implies for forecast confidence
  3. Single most important monitoring KPI with update cadence
  4. Primary risk to the forecast
  5. Model caveat (carry-forward limitation, plain language)
  6. Recommendation for planning robustness (e.g., scenario band rather than point estimate)
"""

# ---------------------------------------------------------------------------
# Print full prompt before sending to the LLM
# ---------------------------------------------------------------------------
SEP = '=' * 72
print(SEP)
print('SYSTEM PROMPT')
print(SEP)
print(SYSTEM_PROMPT)
print()
print(SEP)
print('USER PROMPT')
print(SEP)
print(USER_PROMPT)
print(SEP)
print(f'Model:         {MODEL}')
print(f'Max tokens:    {MAX_TOKENS}')
print(f'Prompt length: {len(USER_PROMPT):,} characters')
print(SEP)
print()

# ---------------------------------------------------------------------------
# API call (streaming)
# ---------------------------------------------------------------------------
print(f'Calling {MODEL} ...\n')
print('-' * 72)

llm_response = ''
with client.messages.stream(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[{'role': 'user', 'content': USER_PROMPT}],
) as stream:
    for text in stream.text_stream:
        llm_response += text
        print(text, end='', flush=True)

print('\n' + '-' * 72)
print('Streaming complete.')

In [ ]:
# Render response as formatted Markdown in the notebook
display(Markdown(llm_response))

# Save to output directory (consistent with scripts/run_defense_analysis.py)
output_path = OUTPUT_DIR / 'defense_analysis_2026.md'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write('# LLM Macroeconomic Analysis — US Defense Market 2026\n\n')
    f.write(f'**Model:** {MODEL}  \n')
    f.write(f'**Prompt length:** {len(USER_PROMPT):,} characters  \n\n')
    f.write('---\n\n')
    f.write(llm_response)

print(f'Response saved to: {output_path}')